# Продуктовая аналитика

**Трек А (командный)**

Датасеты:

*Brazilian E-Commerce Public Dataset by Olist*

*Marketing Funnel by Olist*

### Команда

| №   | Участник         | Пункты ТЗ | Задачи                                                                                            |
| --- | ---------------- | --------- | ------------------------------------------------------------------------------------------------- |
| 1   | Кошелева А. И.   | 3, 8, 9   | Сетап репозитория, метрик и их определений                                                        |
| 2   | Пасхальная Д. А. | 2, 6      | Проверка качества датасета, лицензии, объяснения почему подходит датасет, в 6 когортный retention |
| 3   | Ушинский Д. В.   | 4, 5      | Определение метрик и трекинга                                                                     |
| 4   | Иванова Т. Ю.    | 1, 7      | Описание продукта, этический чек-лист по ТЗ, валидация участника 3                                |
| 5   | Гайсин М. В.     | 2, 6      | Работа со 2 участником, в 6 LTV/CAC и метрики по выбору, по 2 валидация                           |

Связки - 2-5 участник, 3-4 участник

### Оглавление
1. Продукт
2. Данные
3. Определения
4. Метрики
5. Трекинг
6. Первые расчеты
7. Этика
8. Критерии успеха
9. Репозиторий

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA_DIR = Path('data')
assert DATA_DIR.is_dir(), 'Запустите ноутбук из корня репозитория'
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)

print('Папка с данными:', DATA_DIR)

Папка с данными: data


## 1 Продукт

### 1.1 Что делаем

Мы работаем как команда продуктовой аналитики Olist, бразильского e-commerce-сервиса и SaaS-экосистемы.

Olist позволяет малому и среднему бизнесу удобно управлять продажей своих товаров - создавать единый кабинет для управления листингами на всех крупнейших маркетплейсах с одного дашборда. Также в сервис интегрированы инструменты для автоматизации контроля остатков, отслеживания заказов, управления финансами и онлайн-оплаты, упрощенного создания сайтов и отправки заказов через партнерские компании.

 Мы анализируем платформу как двустороннюю площадку (покупатели и продавцы) и выстраиваем систему метрик для понимания точек роста и сильных мест компании.

Основной датасет описывает заказы, позиции, платежи, отзывы, покупателей и продавцов. Второй датасет (*Marketing Funnel*) описывает привлечение продавцов: лиды и закрытые сделки. Датасеты связаны по `seller_id`.


### 1.2 Для кого

| Сегмент | Описание | Цели                                                                                          |
|---|------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------|
| Покупатели | Частные покупатели в Бразилии, которые заказывают товары онлайн | Получить заказ в обещанный срок, нужного качества                                             |
| Продавцы | Малый и средний бизнес, которому нужен канал продаж без собственной инфраструктуры | Получать заказы, не выстраивая свою логистику и платежную систему                             |
| Платформа | Команда Olist                                           | Растить число успешных заказов и удерживать покупателей и продавцов |

### 1.3 Какую проблему решает продукт

| Сегмент | Проблема пользователя                                                                                                                                                                                                                                          | Как продукт ее решает                                                                                                                               |
|---|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------|
| Покупатели | Заказывать у незнакомого небольшого магазина рискованно: непонятно, придет ли товар, когда и в каком виде, к кому обращаться в случае проблем. У каждого магазина свои цены, сроки и условия оплаты. Нужно сравнивать множество сайтов без удобного интерфейса | Единая витрина, единые условия оплаты (в том числе в рассрочку) и доставки, отзывы других покупателей, известно кто отвечает за те или иные проблемы |
| Продавцы | У малого бизнеса нет собственного трафика, систем логистики и приема платежей, а выстраивать все это с нуля долго и дорого                                                                                                                                     | Простое подключение к платформе, где продавец размещает товары, а оплату и доставку обеспечивают платформа и ее партнеры               |

### 1.4 Гипотезы ценности

**Гипотеза ценности** - это проверяемое предположение о том, с какой реальной проблемой (или потребностью) сталкиваются пользователи и как именно ваш продукт или функция способны ее решить, принося им пользу

**H1. Покупатели.** Мы считаем, что покупатели, которые хотят купить у небольших магазинов, но опасаются ненадежности, получают от Olist ценность, когда заказ приходит в срок и соответствует ожиданиям. Это проявится в том, что большинство заказов будут доставлены до обещанной даты и получат оценку 4–5, а часть покупателей вернется за новым заказом.

**H2. Продавцы.** Мы считаем, что малый бизнес без собственного трафика и логистики получает от Olist ценность, когда после подключения к системе начинает получать заказы. Это проявится в том, что _большинство_ продавцов платформы сделают первые продажи за _разумный_ срок и продолжат продавать.

### 1.5 Бизнес-модель

Olist использует комбинированную схему получения дохода:
- Комиссия с продаж (Take Rate): платформа берет процент с каждой успешной транзакции
- SaaS-подписка: продавцы платят фиксированную ежемесячную или ежегодную плату за использование ERP-платформы и инструментов
- Партнерские сервисы: доход от предоставления рекламных инструментов внутри платформы, финансовых услуг и логистических сервисов

### 1.6 Репозиторий

Ноутбук лежит в репозитории: https://github.com/iverid/ProdAnal

## 2. Данные

*Ответственные: участники 2 и 5.*

**TODO:**
- [ ] Источник, лицензия, объем, период, ограничения (версии обоих датасетов).
- [ ] Проверка качества: пропуски, дубли, выбросы, тестовые пользователи, границы времени.
- [ ] Объединение с Marketing Funnel по `seller_id` и оценка охвата.
- [ ] Объяснение, почему датасет подходит.
- [ ] Данные лежат в репозитории (если позволяет лицензия).

*Подготовка данных, аудит и объединение: Пасхальная Д. А.; независимая валидация: Гайсин М. В.*

### 2.1 Источники и воспроизводимость

| Источник (автор — Olist) | Версия | Содержимое локальной выгрузки |
|---|---|---|
| [Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) | 2, обновление 01.10.2021 | 8 CSV: заказы, покупатели, позиции, платежи, отзывы, товары, продавцы, перевод категорий |
| [Marketing Funnel by Olist](https://www.kaggle.com/datasets/olistbr/marketing-funnel-olist) | 2, обновление 16.11.2018 | 2 CSV: MQL и закрытые сделки |

Версии и лицензии сверены с [метаданными e-commerce](https://www.kaggle.com/api/v1/datasets/view/olistbr/brazilian-ecommerce) и [воронки](https://www.kaggle.com/api/v1/datasets/view/olistbr/marketing-funnel-olist).

Оба набора опубликованы под **CC BY-NC-SA 4.0**: требуется атрибуция, некоммерческое использование и та же лицензия для производных материалов. Текст лицензии — в `LICENSE.md`, атрибуция — в README. 


In [ ]:
FILES = {
    'customers': 'Olist E-Commerce Public Dataset/olist_customers_dataset.csv',
    'orders': 'Olist E-Commerce Public Dataset/olist_orders_dataset.csv',
    'items': 'Olist E-Commerce Public Dataset/olist_order_items_dataset.csv',
    'payments': 'Olist E-Commerce Public Dataset/olist_order_payments_dataset.csv',
    'reviews': 'Olist E-Commerce Public Dataset/olist_order_reviews_dataset.csv',
    'products': 'Olist E-Commerce Public Dataset/olist_products_dataset.csv',
    'sellers': 'Olist E-Commerce Public Dataset/olist_sellers_dataset.csv',
    'categories': 'Olist E-Commerce Public Dataset/product_category_name_translation.csv',
    'mql': 'Marketing Funnel by Olist/olist_marketing_qualified_leads_dataset.csv',
    'deals': 'Marketing Funnel by Olist/olist_closed_deals_dataset.csv',
}
raw = {}
inventory = []
for name, relative_path in FILES.items():
    path = DATA_DIR / relative_path
    raw[name] = pd.read_csv(path, dtype={
        'customer_zip_code_prefix': 'string', 'seller_zip_code_prefix': 'string',
    })
    inventory.append({
        'table': name, 'rows': len(raw[name]), 'columns': len(raw[name].columns),
        'bytes': path.stat().st_size, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    })
inventory = pd.DataFrame(inventory).set_index('table')
with pd.option_context('display.max_colwidth', None):
    display(inventory)
print('Размер CSV, MiB:', round(inventory['bytes'].sum() / 1024**2, 2))


,rows,columns,bytes,sha256
table,,,,
customers,99441,5,9033957,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2
orders,99441,8,17654914,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf
items,112650,7,15438671,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279
payments,103886,5,5777138,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0
reviews,99224,7,14451670,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7
products,32951,9,2379446,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc
sellers,3095,4,174703,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e
categories,71,2,2613,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278
mql,8000,4,703836,5537753b7549ce4068e52e2402c29f2773c5375fc0575520ae8f3f301f666573


Размер CSV, MiB: 62.74


### 2.2 Пропуски, дубли и связи

Повтор `order_id` в позициях/платежах/отзывах ожидаем. Повтор `customer_unique_id` означает возвращение покупателя, а не ошибку. Поэтому сначала проверяем полные дубли и ключи каждой сущности. Необязательные комментарии, неизвестные характеристики лида и отсутствующие даты незавершённых этапов не заполняем нулями.

In [3]:
KEYS = {
    'customers': ['customer_id'], 'orders': ['order_id'],
    'items': ['order_id', 'order_item_id'], 'payments': ['order_id', 'payment_sequential'],
    'reviews': ['review_id', 'order_id'], 'products': ['product_id'],
    'sellers': ['seller_id'], 'categories': ['product_category_name'],
    'mql': ['mql_id'], 'deals': ['mql_id'],
}
quality = pd.DataFrame([{
    'table': name, 'full_duplicates': int(df.duplicated().sum()),
    'duplicate_keys': int(df.duplicated(KEYS[name]).sum()),
    'missing_keys': int(df[KEYS[name]].isna().any(axis=1).sum()),
} for name, df in raw.items()]).set_index('table')
missingness = pd.DataFrame([
    {'table': name, 'column': col, 'missing': int(df[col].isna().sum()),
     'missing_pct': round(df[col].isna().mean() * 100, 2)}
    for name, df in raw.items() for col in df if df[col].isna().any()
])
display(quality, missingness)
tables = {name: df.drop_duplicates().copy() for name, df in raw.items()}
for name, df in tables.items():
    assert not df[KEYS[name]].isna().any().any(), (name, 'пропуск ключа')
    assert not df.duplicated(KEYS[name]).any(), (name, 'конфликт ключа')
relations = [
    ('orders', 'customer_id', 'customers', 'customer_id'),
    ('items', 'order_id', 'orders', 'order_id'),
    ('items', 'product_id', 'products', 'product_id'),
    ('items', 'seller_id', 'sellers', 'seller_id'),
    ('payments', 'order_id', 'orders', 'order_id'),
    ('reviews', 'order_id', 'orders', 'order_id'),
    ('deals', 'mql_id', 'mql', 'mql_id'),
    ('deals', 'seller_id', 'sellers', 'seller_id'),
    ('products', 'product_category_name', 'categories', 'product_category_name'),
]
foreign_keys = pd.DataFrame([{
    'relation': f'{child}.{col} → {parent}.{parent_col}',
    'unmatched_rows': int((tables[child][col].notna() &
                           ~tables[child][col].isin(tables[parent][parent_col])).sum()),
} for child, col, parent, parent_col in relations])
display(foreign_keys)
display(pd.DataFrame({
    name: {'unique_order_id': df.order_id.nunique(),
           'orders_without_rows': int((~tables['orders'].order_id.isin(df.order_id)).sum())}
    for name, df in tables.items() if 'order_id' in df and name != 'orders'
}).T)


,full_duplicates,duplicate_keys,missing_keys
table,,,
customers,0,0,0
orders,0,0,0
items,0,0,0
payments,0,0,0
reviews,0,0,0
products,0,0,0
sellers,0,0,0
categories,0,0,0
mql,0,0,0


,table,column,missing,missing_pct
0,orders,order_approved_at,160,0.16
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_delivered_customer_date,2965,2.98
3,reviews,review_comment_title,87656,88.34
4,reviews,review_comment_message,58247,58.70
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01


,relation,unmatched_rows
0,orders.customer_id → customers.customer_id,0
1,items.order_id → orders.order_id,0
2,items.product_id → products.product_id,0
3,items.seller_id → sellers.seller_id,0
4,payments.order_id → orders.order_id,0
5,reviews.order_id → orders.order_id,0
6,deals.mql_id → mql.mql_id,0
7,deals.seller_id → sellers.seller_id,462
8,products.product_category_name → categories.pr...,13


,unique_order_id,orders_without_rows
items,98666,775
payments,99440,1
reviews,98673,768


### 2.3 Период, порядок событий и числовые аномалии

Время разбираем явно, без приписывания часового пояса: его нет в CSV. Проверяем все временные поля, включая `order_approved_at`. В числах показываем диапазон, нули, отрицательные значения.

In [4]:
DATE_COLUMNS = {
    'orders': ['order_purchase_timestamp', 'order_approved_at',
               'order_delivered_carrier_date', 'order_delivered_customer_date',
               'order_estimated_delivery_date'],
    'items': ['shipping_limit_date'],
    'reviews': ['review_creation_date', 'review_answer_timestamp'],
    'mql': ['first_contact_date'], 'deals': ['won_date'],
}
date_audit = []
for name, columns in DATE_COLUMNS.items():
    for col in columns:
        original = tables[name][col]
        parsed = pd.to_datetime(original, errors='coerce')
        invalid = int((original.notna() & parsed.isna()).sum())
        date_audit.append({'table': name, 'column': col, 'min': parsed.min(),
                           'max': parsed.max(), 'invalid': invalid, 'missing': int(parsed.isna().sum())})
        assert invalid == 0, (name, col, 'неразобранные даты')
        tables[name][col] = parsed
orders, items, payments = (tables[name] for name in ['orders', 'items', 'payments'])
customers, sellers, products = (tables[name] for name in ['customers', 'sellers', 'products'])
mql, deals = tables['mql'], tables['deals']
display(pd.DataFrame(date_audit))
display(orders.groupby(orders.order_purchase_timestamp.dt.to_period('M')).agg(
    orders=('order_id', 'size'), delivered=('order_status', lambda x: x.eq('delivered').sum()),
))
deal_dates = deals.merge(mql[['mql_id', 'first_contact_date']], on='mql_id', validate='one_to_one')
time_checks = {
    'approval_before_purchase': orders.order_approved_at.lt(orders.order_purchase_timestamp).sum(),
    'carrier_before_purchase': orders.order_delivered_carrier_date.lt(orders.order_purchase_timestamp).sum(),
    'delivery_before_purchase': orders.order_delivered_customer_date.lt(orders.order_purchase_timestamp).sum(),
    'delivery_before_carrier': orders.order_delivered_customer_date.lt(orders.order_delivered_carrier_date).sum(),
    'delivered_without_delivery_date': (orders.order_status.eq('delivered') & orders.order_delivered_customer_date.isna()).sum(),
    'shipping_limit_from_2019': items.shipping_limit_date.ge('2019-01-01').sum(),
    'deal_before_first_contact': deal_dates.won_date.lt(deal_dates.first_contact_date).sum(),
}
display(pd.Series(time_checks, name='flagged_rows').to_frame())
numeric_columns = {
    'items': ['price', 'freight_value'],
    'payments': ['payment_value', 'payment_installments'],
    'products': ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'],
    'reviews': ['review_score'],
    'deals': ['declared_monthly_revenue', 'declared_product_catalog_size'],
}
numeric_audit = []
for name, columns in numeric_columns.items():
    for col in columns:
        s = tables[name][col]
        q1, q3 = s.quantile([0.25, 0.75])
        numeric_audit.append({
            'table': name, 'column': col, 'min': s.min(), 'median': s.median(),
            'p99': s.quantile(0.99), 'max': s.max(), 'negative': int(s.lt(0).sum()),
            'zero': int(s.eq(0).sum()), 'above_IQR': int(s.gt(q3 + 1.5 * (q3 - q1)).sum()),
        })
display(pd.DataFrame(numeric_audit))
assert tables['reviews'].review_score.between(1, 5).all()


,table,column,min,max,invalid,missing
0,orders,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0,0
1,orders,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,0,160
2,orders,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,0,1783
3,orders,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,0,2965
4,orders,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0,0
5,items,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,0,0
6,reviews,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0,0
7,reviews,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0,0
8,mql,first_contact_date,2017-06-14 00:00:00,2018-05-31 00:00:00,0,0
9,deals,won_date,2017-12-05 02:00:00,2018-11-14 18:04:19,0,0


,orders,delivered
order_purchase_timestamp,,
2016-09,4,1
2016-10,324,265
2016-12,1,1
2017-01,800,750
2017-02,1780,1653
2017-03,2682,2546
2017-04,2404,2303
2017-05,3700,3546
2017-06,3245,3135


,flagged_rows
approval_before_purchase,0
carrier_before_purchase,166
delivery_before_purchase,0
delivery_before_carrier,23
delivered_without_delivery_date,8
shipping_limit_from_2019,4
deal_before_first_contact,1


,table,column,min,median,p99,max,negative,zero,above_IQR
0,items,price,0.85,74.99,890.0000,6735.00,0,0,8427
1,items,freight_value,0.00,16.26,84.5200,409.68,0,383,11613
2,payments,payment_value,0.00,100.00,1039.9165,13664.08,0,9,7981
3,payments,payment_installments,0.00,1.00,10.0000,24.00,0,2,6313
4,products,product_weight_g,0.00,700.00,22538.0000,40425.00,0,4,4551
5,products,product_length_cm,7.00,25.00,100.0000,105.00,0,0,1380
6,products,product_height_cm,2.00,13.00,69.0000,105.00,0,0,1892
7,products,product_width_cm,6.00,20.00,63.0000,118.00,0,0,912
8,reviews,review_score,1.00,5.00,5.0000,5.00,0,0,0
9,deals,declared_monthly_revenue,0.00,0.00,191800.0000,50000000.00,0,797,45


### 2.4 Тестовые пользователи и правила подготовки

Проверяем доступные ID на пустоту и формат 32 шестнадцатеричных символов; корректный формат не доказывает, что пользователь реальный.

Для retention учитываем заказы со статусом `delivered`, с валидной датой покупки и известным `customer_unique_id`. Активность датируем созданием заказа, а не доставкой. 

Отчётное окно — 01.01.2017–31.08.2018, правая граница исключающая 2018-09-01. В 2016 году мало заказов и есть пустой месяц, после августа 2018 — редкий хвост. Эти записи не объявляем ошибками. Для поиска первой покупки сохраняем всю предысторию до правой границы: покупателя из 2016 года нельзя ошибочно сделать новым в 2017. Наличие всех дней в календарном окне не гарантирует полноту исходной выборки; конец августа также может быть усечён.

In [ ]:
identifier_checks = pd.DataFrame([
    {'table': name, 'column': col,
     'missing': int(df[col].isna().sum()),
     'invalid_format': int((df[col].notna() & ~df[col].astype('string').str.fullmatch(r'[0-9a-f]{32}', na=False)).sum())}
    for name, df in tables.items()
    for col in ['customer_id', 'customer_unique_id', 'seller_id', 'mql_id'] if col in df
])
display(identifier_checks)
ANALYSIS_START = pd.Timestamp('2017-01-01')
ANALYSIS_END = pd.Timestamp('2018-09-01')  
LAST_OBSERVED_DAY = ANALYSIS_END - pd.Timedelta(days=1)
order_customers = orders.merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left', validate='one_to_one',
)
assert order_customers.customer_unique_id.notna().all()
valid_activity = (order_customers.order_status.eq('delivered') &
                  order_customers.order_purchase_timestamp.notna() &
                  order_customers.customer_unique_id.notna())
purchase_history = order_customers.loc[
    valid_activity & order_customers.order_purchase_timestamp.lt(ANALYSIS_END)
].copy()
purchase_history['purchase_date'] = purchase_history.order_purchase_timestamp.dt.normalize()
analysis_orders = purchase_history.loc[purchase_history.purchase_date.ge(ANALYSIS_START)].copy()
display(pd.Series({
    'raw_orders': len(orders),
    'not_delivered': int((~orders.order_status.eq('delivered')).sum()),
    'delivered_history_before_window': int(purchase_history.purchase_date.lt(ANALYSIS_START).sum()),
    'delivered_on_or_after_end': int((valid_activity & order_customers.order_purchase_timestamp.ge(ANALYSIS_END)).sum()),
    'analysis_orders': len(analysis_orders),
    'analysis_buyers': analysis_orders.customer_unique_id.nunique(),
}, name='count').to_frame())
item_totals = items.groupby('order_id', as_index=False).agg(
    item_count=('order_item_id', 'size'), seller_count=('seller_id', 'nunique'),
    item_gmv=('price', 'sum'), freight_value=('freight_value', 'sum'),
)
payment_totals = payments.groupby('order_id', as_index=False).agg(payment_value=('payment_value', 'sum'))
order_facts = order_customers.merge(item_totals, on='order_id', how='left', validate='one_to_one').merge(
    payment_totals, on='order_id', how='left', validate='one_to_one',
)
assert len(order_facts) == len(orders) and order_facts.order_id.is_unique
assert np.isclose(order_facts.item_gmv.sum(), items.price.sum())
assert np.isclose(order_facts.payment_value.sum(), payments.payment_value.sum())
print('order_facts: одна строка на заказ; GMV не является выручкой платформы.')


,table,column,missing,invalid_format
0,customers,customer_id,0,0
1,customers,customer_unique_id,0,0
2,orders,customer_id,0,0
3,items,seller_id,0,0
4,sellers,seller_id,0,0
5,mql,mql_id,0,0
6,deals,seller_id,0,0
7,deals,mql_id,0,0


,count
raw_orders,99441
not_delivered,2963
delivered_history_before_window,267
delivered_on_or_after_end,0
analysis_orders,96211
analysis_buyers,93104


order_facts: одна строка на заказ; GMV не является выручкой платформы.


### 2.5 Объединение с воронкой 

Funnel относится к продавцам, а не к привлечению покупателей. 


In [6]:
assert deals.seller_id.is_unique, 'Нужно согласовать правило атрибуции нескольких MQL продавцу'
funnel = mql.merge(deals, on='mql_id', how='left', validate='one_to_one', indicator='deal_match')
deal_mql = deals.merge(mql, on='mql_id', how='left', validate='one_to_one')
seller_funnel = sellers.merge(deal_mql, on='seller_id', how='left', validate='one_to_one')
items_with_funnel = items.merge(deal_mql, on='seller_id', how='left', validate='many_to_one').merge(
    orders[['order_id', 'order_purchase_timestamp', 'order_status']],
    on='order_id', how='left', validate='many_to_one',
)
items_with_funnel['funnel_known_at_purchase'] = (
    items_with_funnel.won_date.notna() &
    items_with_funnel.won_date.le(items_with_funnel.order_purchase_timestamp)
)
assert len(funnel) == len(mql)
assert len(seller_funnel) == len(sellers)
assert len(items_with_funnel) == len(items)
assert np.isclose(items_with_funnel.price.sum(), items.price.sum())
matched_sellers = sellers.seller_id.isin(deals.seller_id).sum()
matched_items = items_with_funnel.mql_id.notna()
coverage = pd.DataFrame([
    ('MQL с закрытой сделкой', len(deals), len(mql)),
    ('Продавцы e-commerce с MQL', matched_sellers, len(sellers)),
    ('Сделки с продавцом в e-commerce', matched_sellers, len(deals)),
    ('Позиции с привязанным MQL', int(matched_items.sum()), len(items)),
    ('Заказы хотя бы с одним MQL-продавцом', items_with_funnel.loc[matched_items, 'order_id'].nunique(), len(orders)),
    ('Позиции с известной на дату покупки сделкой', int(items_with_funnel.funnel_known_at_purchase.sum()), len(items)),
], columns=['coverage', 'matched', 'base']).set_index('coverage')
coverage['pct'] = (coverage.matched / coverage.base * 100).round(2)
display(coverage)
display(pd.Series({
    'deals_after_analysis_end': int(deals.won_date.ge(ANALYSIS_END).sum()),
    'matched_items_before_deal': int((matched_items & ~items_with_funnel.funnel_known_at_purchase).sum()),
}, name='count').to_frame())


,matched,base,pct
coverage,,,
MQL с закрытой сделкой,842,8000,10.52
Продавцы e-commerce с MQL,380,3095,12.28
Сделки с продавцом в e-commerce,380,842,45.13
Позиции с привязанным MQL,5044,112650,4.48
Заказы хотя бы с одним MQL-продавцом,4512,99441,4.54
Позиции с известной на дату покупки сделкой,5044,112650,4.48


,count
deals_after_analysis_end,50
matched_items_before_deal,0


### 2.6 Вывод

Основной набор подходит для анализа повторных успешных покупок: есть устойчивый customer_unique_id, время создания, статусы и история заказов. Позиции, платежи и доставка позволяют диагностировать качество выполнения заказа. Funnel добавляет канал и дату привлечения части продавцов, но не описывает путь покупателя от визита до покупки.

Ограничения: это анонимизированная историческая выборка; первая наблюдаемая покупка может не быть первой в жизни. Нет сессий, затрат на привлечение, комиссии, маржи, подписок и полных возвратов — фактические CAC и LTV платформы не идентифицируются. 

Полных дублей и конфликтов выбранных ключей в локальных таблицах нет. Пропуски в отзывах и характеристиках лидов многочисленны и не означают отсутствие покупки. 

166 заказов переданы перевозчику раньше создания, 23 доставлены раньше передачи, 8 `delivered` без даты доставки; у 4 позиций дедлайн отправки в 2020 году, у 1 сделки дата победы раньше первого контакта. Для анализа длительности этапов такие записи требуют отдельного правила; в retention используется только валидная дата покупки. 

Воронка покрывает **380 / 3095 = 12.28% продавцов**, **5044 / 112650 = 4.48% позиций**, а не всю платформу. Из 842 сделок 462 продавца не найдены в e-commerce, 50 сделок закрыты уже после отчётного окна. Отсутствие продаж нельзя автоматически считать неуспешным онбордингом.

## 3 Определения

*Ответственный: участник 1.*

**TODO:**
- [ ] Активный пользователь
- [ ] Конверсия
- [ ] Удержание
- [ ] Когорта
- [ ] Правило оттока

### 3.1 Активный пользователь

В данных нет входов и сессий. Единственное подтверждённое действие пользователя - заказ, соответственно активность будет вычисляться через заказы

- Активный покупатель за период N - покупатель, у которого в периоде N есть хотя бы один учитываемый заказ
- Активный продавец за период N - продавец, у которого в периоде N есть хотя бы одна позиция в учитываемом заказе (сделан 1 заказ)

### 3.2 Конверсия

В проекте три конверсии:

- Заказ - доставка: заказы со статусом delivered / все учитываемые заказы периода создания
- Лид - закрытая сделка: лиды с закрытой сделкой / все лиды
- Контракт с продавцом - активный продавец:	активные продавцы / все закрытые сделки

### 3.3 Удержание
- Первый заказ покупателя - самый ранний учитываемый заказ
- Повторный заказ - учитываемый заказ, созданный в более позднюю календарную дату, чем первый. Заказы того же дня считаются частью первой покупки (разбивка одной покупки на несколько)
- Удержание за период N - доля покупателей, у которых есть хотя бы один повторный заказ в течение периода N после даты первого

### 3.4 Когорта
- Когорта покупателей - выполняется по времени первого учитываемого заказа
- Когорта продавцов - выполняется по времени заключения контракта с платформой

### 3.5 Правило оттока
Покупатель ушёл, если с даты его последнего учитываемого заказа прошло N дней без нового заказа. Продавец ушёл, если с даты его последнего заказа прошло M дней без нового.

Числа N, M для правила оттока требуют отдельного согласования.



## 4. Метрики

*Ответственный: участник 3.*

**TODO:**
- [ ] North Star, две диагностические и одна риск-метрика по пирамиде метрик.
- [ ] Для каждой: зачем нужна и какое решение на нее опирается.
- [ ] Свериться с разд. 8: метрики и целевые значения должны совпадать.

## 5. Трекинг

*Ответственный: участник 3.*

**TODO:** таблица минимум из 5 событий.

| Событие | Триггер | Параметры | Метрика | Этап пути | Поле датасета |
|---|---|---|---|---|---|
| | | | | | |

## 6. Первые расчеты

*Ответственные: участник 2, участник 5.*

**TODO:**
- [ ] LTV/CAC со сценариями CAC.
- [ ] Когортный retention.
- [ ] Еще один-два расчета (например, MAU/WAU).
- [ ] Базовые значения метрик для разд. 8.
- [ ] К каждому расчету таблица или график и вывод.

*Когортный retention: Пасхальная Д. А. LTV/CAC и дополнительные метрики: Гайсин М. В.*

### 6.1 Когортный retention покупателей по календарным месяцам

Для когорты c и возраста k ≥ 1: 

retention[c, k] = уникальные покупатели когорты c с повтором в месяце c+k / размер когорты c.

In [7]:
first_purchase = purchase_history.groupby('customer_unique_id')['purchase_date'].min().rename('first_purchase_date')
buyer_cohorts = first_purchase.to_frame()
buyer_cohorts['cohort'] = buyer_cohorts.first_purchase_date.dt.to_period('M')
buyer_cohorts = buyer_cohorts.loc[buyer_cohorts.first_purchase_date.ge(ANALYSIS_START)].copy()
cohort_index = pd.period_range(ANALYSIS_START, LAST_OBSERVED_DAY, freq='M')
cohort_sizes = buyer_cohorts.groupby('cohort').size().reindex(cohort_index, fill_value=0).rename('buyers')
cohort_sizes.index.name = 'cohort'
buyer_activity = purchase_history.merge(
    buyer_cohorts, left_on='customer_unique_id', right_index=True, how='inner', validate='many_to_one',
)
buyer_activity['age_days'] = (buyer_activity.purchase_date - buyer_activity.first_purchase_date).dt.days
buyer_activity['activity_month'] = buyer_activity.purchase_date.dt.to_period('M')
buyer_activity['month_age'] = (
    buyer_activity.activity_month.astype('int64') - buyer_activity.cohort.astype('int64')
)
repeats = buyer_activity.loc[buyer_activity.age_days.gt(0)].copy()
month_ages = pd.Index(range(len(cohort_index)), name='month_age')
monthly_counts = repeats.groupby(['cohort', 'month_age']).customer_unique_id.nunique().unstack(
    'month_age', fill_value=0,
).reindex(index=cohort_index, columns=month_ages, fill_value=0)
monthly_counts[0] = cohort_sizes  # M0 — приобретение, а не повтор
observed_months = pd.DataFrame(
    [[cohort + int(k) <= LAST_OBSERVED_DAY.to_period('M') for k in month_ages] for cohort in cohort_index],
    index=cohort_index, columns=month_ages,
)
monthly_counts = monthly_counts.where(observed_months & cohort_sizes.gt(0).to_numpy()[:, None])
monthly_retention = monthly_counts.div(cohort_sizes.replace(0, np.nan), axis=0)
calendar_table = monthly_retention.iloc[:, :13].mul(100).round(2)
calendar_table.columns = [f'M{k}, %' for k in calendar_table.columns]
calendar_table.insert(0, 'buyers', cohort_sizes)
display(calendar_table.astype(object).fillna('—'))
print('Показаны M0–M12; полные матрицы monthly_counts / monthly_retention содержат все доступные месяцы.')
print('Покупателей с первой покупкой до 2017, активных в окне:',
      analysis_orders.loc[analysis_orders.customer_unique_id.isin(
          first_purchase.index[first_purchase.lt(ANALYSIS_START)]
      ), 'customer_unique_id'].nunique())


,buyers,"M0, %","M1, %","M2, %","M3, %","M4, %","M5, %","M6, %","M7, %","M8, %","M9, %","M10, %","M11, %","M12, %"
2017-01,717,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14,0.14,0.0,0.42,0.14,0.7
2017-02,1628,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18,0.12,0.18,0.12,0.31,0.12
2017-03,2503,100.0,0.44,0.36,0.4,0.36,0.16,0.16,0.32,0.32,0.08,0.36,0.12,0.2
2017-04,2256,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31,0.31,0.18,0.27,0.09,0.04
2017-05,3451,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14,0.26,0.26,0.26,0.35,0.23
2017-06,3037,100.0,0.49,0.4,0.43,0.3,0.4,0.36,0.23,0.13,0.2,0.3,0.36,0.16
2017-07,3752,100.0,0.53,0.35,0.24,0.29,0.21,0.32,0.11,0.19,0.27,0.21,0.29,0.13
2017-08,4057,100.0,0.69,0.35,0.27,0.35,0.52,0.3,0.27,0.15,0.15,0.25,0.2,0.12
2017-09,4004,100.0,0.7,0.55,0.27,0.45,0.22,0.22,0.25,0.27,0.17,0.25,0.07,—
2017-10,4328,100.0,0.72,0.25,0.09,0.23,0.21,0.21,0.37,0.28,0.18,0.21,—,—


Показаны M0–M12; полные матрицы monthly_counts / monthly_retention содержат все доступные месяцы.
Покупателей с первой покупкой до 2017, активных в окне: 10


### 6.2 Накопленный возврат в течение 30, 60 и 90 дней

R_H = покупатели с повтором на дне 1 ≤ d ≤ H / все покупатели когорт.

In [8]:
first_repeat_age = repeats.groupby('customer_unique_id').age_days.min()
buyer_cohorts['first_repeat_age'] = first_repeat_age
HORIZONS = (30, 60, 90)
horizon_counts = pd.DataFrame(index=cohort_index)
horizon_observed = pd.DataFrame(index=cohort_index)
for horizon in HORIZONS:
    mature = cohort_index.to_timestamp(how='end').normalize() + pd.Timedelta(days=horizon) <= LAST_OBSERVED_DAY
    returned = buyer_cohorts.loc[buyer_cohorts.first_repeat_age.between(1, horizon)].groupby('cohort').size()
    horizon_counts[horizon] = returned.reindex(cohort_index, fill_value=0).where(mature & cohort_sizes.gt(0))
    horizon_observed[horizon] = mature & cohort_sizes.gt(0)
horizon_retention = horizon_counts.div(cohort_sizes.replace(0, np.nan), axis=0)
horizon_table = pd.DataFrame({'buyers': cohort_sizes})
for horizon in HORIZONS:
    horizon_table[f'returned_{horizon}'] = horizon_counts[horizon]
    horizon_table[f'R{horizon}, %'] = horizon_retention[horizon].mul(100).round(2)
display(horizon_table.astype(object).fillna('—'))
summary_rows = []
for horizon in HORIZONS:
    mature = horizon_observed[horizon]
    denominator = int(cohort_sizes.loc[mature].sum())
    numerator = int(horizon_counts.loc[mature, horizon].sum())
    summary_rows.append({'metric': f'R{horizon}', 'cohorts': int(mature.sum()),
                         'buyers': denominator, 'returned': numerator,
                         'retention_pct': 100 * numerator / denominator})
m1_mature = observed_months[1] & cohort_sizes.gt(0)
summary_rows.append({'metric': 'M1', 'cohorts': int(m1_mature.sum()),
                     'buyers': int(cohort_sizes.loc[m1_mature].sum()),
                     'returned': int(monthly_counts.loc[m1_mature, 1].sum()),
                     'retention_pct': 100 * monthly_counts.loc[m1_mature, 1].sum() / cohort_sizes.loc[m1_mature].sum()})
retention_summary = pd.DataFrame(summary_rows).set_index('metric')
display(retention_summary.round(3))
common_mature = horizon_observed[90]
common_base = int(cohort_sizes.loc[common_mature].sum())
common_horizon_summary = pd.DataFrame({
    'buyers': common_base,
    'returned': horizon_counts.loc[common_mature].sum().astype(int),
})
common_horizon_summary.index = [f'R{h}' for h in HORIZONS]
common_horizon_summary['retention_pct'] = common_horizon_summary.returned / common_base * 100
print('Одна и та же база когорт с полным 90-дневным наблюдением:')
display(common_horizon_summary.round(3))


,buyers,returned_30,"R30, %",returned_60,"R60, %",returned_90,"R90, %"
cohort,,,,,,,
2017-01,717,4.0,0.56,6.0,0.84,6.0,0.84
2017-02,1628,10.0,0.61,12.0,0.74,15.0,0.92
2017-03,2503,12.0,0.48,19.0,0.76,32.0,1.28
2017-04,2256,16.0,0.71,24.0,1.06,30.0,1.33
2017-05,3451,31.0,0.9,43.0,1.25,55.0,1.59
2017-06,3037,17.0,0.56,30.0,0.99,43.0,1.42
2017-07,3752,29.0,0.77,42.0,1.12,49.0,1.31
2017-08,4057,25.0,0.62,50.0,1.23,58.0,1.43
2017-09,4004,37.0,0.92,55.0,1.37,73.0,1.82


,cohorts,buyers,returned,retention_pct
metric,,,,
R30,19,86950,576,0.662
R60,18,81001,829,1.023
R90,17,75123,976,1.299
M1,19,86950,420,0.483


Одна и та же база когорт с полным 90-дневным наблюдением:


,buyers,returned,retention_pct
R30,75123,500,0.666
R60,75123,778,1.036
R90,75123,976,1.299


In [ ]:
m1 = retention_summary.loc['M1']
r30 = retention_summary.loc['R30']
r90 = retention_summary.loc['R90']
display(Markdown(
    f'**Вывод.** В зрелых когортах M1 = **{m1.retention_pct:.3f}%** '
    f'({int(m1.returned)} / {int(m1.buyers)} покупателей). '
    f'В течение 30 дней вернулись **{r30.retention_pct:.3f}%** '
    f'({int(r30.returned)} / {int(r30.buyers)}), в течение 90 дней — **{r90.retention_pct:.3f}%** '
    f'({int(r90.returned)} / {int(r90.buyers)}). '
    'Повторные покупки в выборке редки; малые различия между когортами могут соответствовать '
    'единицам покупателей и не доказывают улучшение продукта. Это удержание по успешным покупкам, '
    'а не посещаемость сервиса. Последние когорты имеют меньше наблюдаемой истории, поэтому '
    'прочерки нельзя интерпретировать как отток. Для решения о стимулировании второй покупки '
    'имеет смысл проверить причины невозврата и тестировать коммуникации на горизонте 30–90 дней; '
    'сами наблюдения не устанавливают причинный эффект.'
))


**Вывод.** В зрелых когортах M1 = **0.483%** (420 / 86950 покупателей). В течение 30 дней вернулись **0.662%** (576 / 86950), в течение 90 дней — **1.299%** (976 / 75123). Повторные покупки в выборке редки; малые различия между когортами могут соответствовать единицам покупателей и не доказывают улучшение продукта. Это удержание по успешным покупкам, а не посещаемость сервиса. Последние когорты имеют меньше наблюдаемой истории, поэтому прочерки нельзя интерпретировать как отток. Для решения о стимулировании второй покупки имеет смысл проверить причины невозврата и тестировать коммуникации на горизонте 30–90 дней; сами наблюдения не устанавливают причинный эффект.

## 7. Этика

*Ответственный: участник 4.*

**TODO:** чек-лист.
- [ ] Цель
- [ ] Согласие
- [ ] Специальные категории данных
- [ ] Передача третьим лицам
- [ ] Хранение и удаление
- [ ] Лицензия и происхождение датасета (CC BY-NC-SA 4.0)

## 8. Критерии успеха

Будет видно по метрикам определенным на этапе 4

## 9. Репозиторий

*Ответственный: участник 1.*

**TODO:**
- [ ] README: как запустить, откуда данные, что получилось, вклад участников.
- [ ] `.gitignore` и `requirements.txt` с зафиксированными версиями.
- [ ] Лицензия (CC BY-NC-SA 4.0) и атрибуция датасетов.
- [ ] Все пути в ноутбуке относительные.
- [ ] Проверка: чистый клон, новое окружение, Restart & Run All.